# Project 3 — Local Meeting Notes Summarizer

## Summarize Transcripts into Actions, Decisions, and Blockers

**Goal:** Take a raw meeting transcript and produce a structured summary with
action items, decisions made, blockers identified, and key discussion points —
all using a local Ollama model.

**Stack:** Ollama · LangChain · Jupyter

### What You'll Learn

1. Design **structured summarization prompts** that extract specific sections
2. Compare **plain summary** vs **structured extraction**
3. Use **Pydantic models** for validated structured output
4. Handle **long transcripts** with chunked summarization
5. Analyze **prompt variant quality**

### Prerequisites

- Ollama running with `qwen3:8b` pulled
- Python 3.9+

In [ ]:
# Install dependencies (uncomment and run once)
# !pip install -q langchain langchain-ollama langchain-core

## Step 1 — Verify Ollama

In [ ]:
import requests
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    r.raise_for_status()
    print(f"Ollama is running — {len(r.json().get('models', []))} model(s)")
except Exception as e:
    print(f"Cannot reach Ollama: {e}\n  Run: ollama pull qwen3:8b")

## Step 2 — Configure LLM

In [ ]:
from langchain_ollama import ChatOllama

llm = ChatOllama(model="qwen3:8b", temperature=0)
resp = llm.invoke("Say 'ready' in one word.")
print(f"LLM ready: {resp.content[:80]}")

## Step 3 — Create a Sample Meeting Transcript

We create a realistic (but synthetic) meeting transcript with multiple speakers,
decisions, action items, and blockers embedded naturally in the conversation.

In [ ]:
transcript = """
Meeting: Q2 Platform Planning
Date: 2025-03-15
Attendees: Sarah (PM), Mike (Engineering Lead), Lisa (Design), Tom (QA)

Sarah: Let's start with the API redesign status. Mike, where are we?

Mike: We've completed the schema migration for the user service. The auth
endpoints are 90% done. We're blocked on the rate limiting implementation
because we need the infrastructure team to provision the Redis cluster.

Sarah: That's a blocker. Tom, can you escalate that to infra?

Tom: I'll file a priority ticket today and follow up by Wednesday.

Lisa: On the design side, the new dashboard mockups are ready for review.
I've shared them in Figma. The mobile responsive version needs another
iteration based on the feedback from the usability study.

Sarah: Great. Let's schedule a design review for Thursday. Mike, can your
team start the frontend implementation next week?

Mike: Yes, but we need the final API contracts first. I propose we freeze
the API spec by Friday so frontend can start Monday.

Sarah: Agreed. Decision: API spec freeze by Friday March 21st.

Tom: For QA, I've set up the automated regression suite for the new endpoints.
We found 3 critical bugs in the batch processing module. Two are fixed, one
needs Mike's team to review.

Mike: I'll assign someone to look at that critical bug tomorrow.

Sarah: Any other blockers?

Lisa: We need the brand team to approve the new color palette. I've been
waiting two weeks. This blocks the final design handoff.

Sarah: I'll escalate that today. Let's wrap up. Next meeting same time next week.
"""

print(f"Transcript: {len(transcript)} characters, ~{len(transcript.split())} words")

## Step 4 — Plain Summary

First, let's see what a simple "summarize this" prompt produces.
This establishes our baseline before trying structured extraction.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

plain_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a meeting assistant. Summarize the following meeting transcript concisely."),
    ("human", "{transcript}"),
])

plain_chain = plain_prompt | llm | StrOutputParser()
plain_summary = plain_chain.invoke({"transcript": transcript})

print("=== Plain Summary ===")
print(plain_summary)

## Step 5 — Structured Summary Extraction

Now let's use a more specific prompt that extracts **distinct sections**:
actions, decisions, blockers, and key points. This is much more useful
for follow-up than a plain paragraph summary.

In [ ]:
structured_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a meeting assistant. Analyze the transcript and extract:

1. **DECISIONS** — What was agreed upon
2. **ACTION ITEMS** — Who will do what, by when
3. **BLOCKERS** — What is preventing progress
4. **KEY DISCUSSION POINTS** — Main topics discussed

Format each section clearly with bullet points. Be specific about owners and deadlines."""),
    ("human", "{transcript}"),
])

structured_chain = structured_prompt | llm | StrOutputParser()
structured_summary = structured_chain.invoke({"transcript": transcript})

print("=== Structured Summary ===")
print(structured_summary)

## Step 6 — JSON Extraction with Validation

For downstream automation (ticketing, email drafts), we want **machine-readable output**.
Let's ask the model to produce JSON and validate the structure.

In [ ]:
import json

json_prompt = ChatPromptTemplate.from_messages([
    ("system", """Extract meeting information as valid JSON with this exact structure:
{{
  "meeting_title": "...",
  "date": "...",
  "decisions": ["decision 1", "decision 2"],
  "action_items": [
    {{"owner": "Name", "task": "description", "deadline": "date or null"}}
  ],
  "blockers": [
    {{"description": "...", "owner": "who is affected", "escalation": "who will escalate"}}
  ],
  "key_topics": ["topic 1", "topic 2"]
}}

Return ONLY valid JSON, no markdown formatting."""),
    ("human", "{transcript}"),
])

json_chain = json_prompt | llm | StrOutputParser()
raw_output = json_chain.invoke({"transcript": transcript})

# Try to parse the JSON
print("=== Raw LLM Output ===")
print(raw_output[:500])

# Clean and parse
try:
    # Strip potential markdown code fences
    cleaned = raw_output.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.split("\n", 1)[1]
    if cleaned.endswith("```"):
        cleaned = cleaned.rsplit("\n", 1)[0]
    # Try to find JSON in output (handle thinking tags)
    if "{" in cleaned:
        json_start = cleaned.index("{")
        json_end = cleaned.rindex("}") + 1
        cleaned = cleaned[json_start:json_end]
    parsed = json.loads(cleaned)
    print("\n=== Parsed JSON ===")
    print(json.dumps(parsed, indent=2))
    print(f"\nExtracted: {len(parsed.get('action_items', []))} action items, "
          f"{len(parsed.get('blockers', []))} blockers, "
          f"{len(parsed.get('decisions', []))} decisions")
except json.JSONDecodeError as e:
    print(f"\nJSON parsing failed: {e}")
    print("This is a common challenge with local models — see Limitations below.")

## Step 7 — Compare Prompt Variants

Different prompt styles produce different quality summaries.
Let's compare a few approaches to see which works best for our model.

In [ ]:
prompt_variants = {
    "bullet_style": "Summarize this meeting as bullet points grouped by topic.",
    "executive": "Write a 3-sentence executive summary of this meeting for a VP.",
    "email_followup": "Draft a follow-up email summarizing this meeting for all attendees. Include action items with owners.",
}

for style, instruction in prompt_variants.items():
    variant_prompt = ChatPromptTemplate.from_messages([
        ("system", instruction),
        ("human", "{transcript}"),
    ])
    result = (variant_prompt | llm | StrOutputParser()).invoke({"transcript": transcript})
    print(f"\n{'='*60}")
    print(f"Style: {style}")
    print(f"{'='*60}")
    print(result[:400])
    print("..." if len(result) > 400 else "")

## Step 8 — Failure Cases

Let's test edge cases: very short input, gibberish, and non-meeting content.

In [ ]:
edge_cases = {
    "very_short": "Bob: Let's meet tomorrow. Alice: OK.",
    "no_actions": "Team discussed the weather and had coffee. No decisions were made.",
    "non_meeting": "The quick brown fox jumps over the lazy dog. Pack my box with five dozen liquor jugs.",
}

for case_name, text in edge_cases.items():
    result = (structured_prompt | llm | StrOutputParser()).invoke({"transcript": text})
    print(f"\n--- {case_name} ---")
    print(f"Input: {text[:60]}...")
    print(f"Output: {result[:200]}...")

## Limitations & Tradeoffs

| Aspect | What happens | How to improve |
|--------|-------------|----------------|
| **JSON reliability** | Local models may produce malformed JSON | Add retry with error feedback |
| **Long transcripts** | May exceed context window | Use map-reduce or chunking |
| **Speaker diarization** | Assumes speakers are labeled in text | Add pre-processing for unlabeled audio |
| **Hallucinated actions** | Model may invent action items | Cross-check against transcript |
| **Thinking tags** | qwen3 may include reasoning | Post-process to strip tags |

### What this project does NOT cover
- Audio transcription (see Project 88)
- Speaker identification from audio
- Integration with task management tools

## What You Learned

1. **Plain vs structured summarization** — structured prompts extract more useful information
2. **JSON extraction** — asking for structured output enables downstream automation
3. **Prompt variants** — different styles serve different audiences (executive, team, email)
4. **Edge case handling** — testing unusual inputs reveals model limitations
5. **Output validation** — parsing and validating LLM output is essential for reliability

## Exercises

1. **Add your own transcript** — paste a real meeting transcript and compare outputs
2. **Add a priority field** — extend the JSON schema to include priority for each action item
3. **Handle long meetings** — split a transcript into 5-minute chunks and merge summaries
4. **Build a manager update** — create a prompt that generates a weekly status email from multiple meeting summaries

---

*Next project: **04 — Local Resume Rewriter** (iterative rewriting with Ollama)*